# Spark X2.6 4B Recursive72 — Loop72 + Paged D-LSA Validation & Benchmark Suite

**Model:** `Spark X2.6 4B Recursive72` (`TassP0lar/Spark-X2.6-4B-Recursive72`)  
**Architecture:** 36 physical layers $\rightarrow$ 72 virtual visits (`4 + 24 + 24 + 12 + 8` = 18 global + 54 local), bounded-gain `TwoStreamMHC` Schema v2 (`gain_mode="sigmoid"`, `mhc_schema_version=2`, `10,078,176` parameters), and 18-visit Stage E `VisitAwareDLSAIndexBank` (`1,769,472` selector parameters).

This notebook separates runtime checks, smoke tasks, inference timing, complete dataset runs, and actual Oh My Pi (`omp`) coding tests. Full dataset adapters cover GraphWalks BFS, MRCR 8-needle, and LongBench v2.

See `docs/roadmap/09_LOOP72_MHC_TRAINING_AND_VALIDATION.md` for the training sequence and release criteria.


In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path

REPO_URL = 'https://github.com/BaiTian6641/Spark-X2.6-Recursion.git'
BRANCH = 'loop72-dlsa'
REPO_DIR = '/content/Spark-X2.6-Recursion' if os.path.exists('/content') else os.getcwd()

if os.path.exists('/content'):
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', '-r', 'requirements-omp.txt'], check=True)

GIT_COMMIT = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
import torch
import sparkx_full_benchmark as full
import sparkx_loop72_dlsa_benchmarks as bench
import load_inference_v03 as loader
from sparkx_omp_server import DEFAULT_MODEL_ID
from train_spark_mhc import DEFAULT_HF_REPO_ID, normalize_hf_repo_id, resolve_hf_token

RUN_STAMP = time.strftime('%Y%m%d_%H%M%S')
RUN_DIR = Path(f'runs/loop72_recursive72_validation_{RUN_STAMP}')
RUN_DIR.mkdir(parents=True, exist_ok=True)
PREFLIGHT = {**full.preflight(), 'git_commit': GIT_COMMIT, 'run_stamp': RUN_STAMP}
(RUN_DIR / 'preflight.json').write_text(json.dumps(PREFLIGHT, indent=2))
print(json.dumps(PREFLIGHT, indent=2))
# Preflight checks prerequisites; it is not a model benchmark.


In [ ]:
# Automatically use local export if present in this runtime, otherwise pull `TassP0lar/Spark-X2.6-4B-Recursive72` from Hugging Face Hub.
HF_TOKEN = resolve_hf_token()
LOCAL_EXPORT_DIR = Path('results/mhc_colab_g4_run/final_loop72_mhc_dlsa_export')
LOCAL_SELECTOR_CKPT = Path('results/mhc_colab_g4_run/visit_selectors_stage_e.pt')

if LOCAL_EXPORT_DIR.is_dir():
    CHECKPOINT_DIR = str(LOCAL_EXPORT_DIR)
    DLSA_CHECKPOINT = str(LOCAL_SELECTOR_CKPT) if LOCAL_SELECTOR_CKPT.is_file() else str(LOCAL_EXPORT_DIR / 'visit_selectors_stage_e.pt')
else:
    CHECKPOINT_DIR = normalize_hf_repo_id(os.environ.get('HF_REPO_ID', DEFAULT_HF_REPO_ID), token=HF_TOKEN)
    DLSA_CHECKPOINT = 'visit_selectors_stage_e.pt'

MODEL_REVISION = full.MODEL_REVISION
VARIANT_REVISION = os.environ.get('VARIANT_REVISION') or None
RECURRENCE_MODE = 'learned_mhc'
print(f'Loading Spark X2.6 4B Recursive72 from: {CHECKPOINT_DIR} (recurrence_mode={RECURRENCE_MODE})')

model, tokenizer = loader.load_sparkx_loop72_lsa(
    repo_or_dir=CHECKPOINT_DIR,
    base_repo='XHToken/Spark-X2.6-4B',
    base_revision=MODEL_REVISION,
    variant_revision=VARIANT_REVISION,
    dlsa_checkpoint=DLSA_CHECKPOINT,
    recurrence_mode=RECURRENCE_MODE,
    strict_selector_provenance=True,
    require_verified_selectors=True,
    token=HF_TOKEN,
)
manifest = model.execution_manifest()
assert manifest['total_virtual_layers'] == 72
assert manifest['global_visits'] == 18 and manifest['local_visits'] == 54
assert manifest['engram_constructed'] is False
assert manifest['effective_recurrence_mode'] == 'learned_mhc'
assert manifest['mhc_gain_mode'] == 'sigmoid'
assert manifest['mhc_schema_version'] == 2
assert manifest['selector_checkpoint_status']['initialization'] in ('checkpoint_loaded_verified', 'loop_state_checkpoint_loaded_verified')
assert manifest['selector_checkpoint_status']['trained_global_visits'] == 18
assert manifest['selector_checkpoint_status']['recurrence_mode_match'] is True
print(json.dumps(manifest, indent=2))
(RUN_DIR / 'model_manifest.json').write_text(json.dumps(manifest, indent=2))


In [ ]:
# Explicit smoke tasks test plumbing; these are not official scores.
SMOKE_REPORT = bench.evaluate_selected_benchmark_tasks(
    model, tokenizer, max_new_tokens=2048,
    include_hf_streamed=False, include_smoke=True, task_mode='diagnostic',
    verbose=True,
)
(RUN_DIR / 'smoke.json').write_text(json.dumps(SMOKE_REPORT, indent=2))
print(json.dumps(SMOKE_REPORT['by_benchmark'], indent=2))


In [ ]:
# Fresh caches, repeated measurements, actual generated counts.
SPEED_REPORT = bench.benchmark_loop72_decode_speed(
    model, tokenizer, prompt_lengths=(512, 2048, 8192, 32768),
    gen_tokens=512, warmups=1, repeats=3, verbose=True,
)
(RUN_DIR / 'speed.json').write_text(json.dumps(SPEED_REPORT, indent=2))
print(json.dumps(SPEED_REPORT['horizons'], indent=2))


In [ ]:
# Full means every eligible row of each pinned dataset, without a sample cap.
RUN_FULL_DATASETS = True
if RUN_FULL_DATASETS:
    sel_status = model.execution_manifest()['selector_checkpoint_status']
    if sel_status.get('initialization') not in ('checkpoint_loaded_verified', 'loop_state_checkpoint_loaded_verified'):
        raise RuntimeError(f'Load a verified 18-visit Stage E D-LSA checkpoint before long-context evaluation (got {sel_status!r}).')
    full_out_dir = RUN_DIR / 'full'
    full_out_dir.mkdir(parents=True, exist_ok=True)
    FULL_REPORT = full.run_full_benchmarks(
        model, tokenizer, full_out_dir, suites=None,
        max_new_tokens=32768, limit=None, verbose=True,
    )
else:
    FULL_REPORT = {'status': 'not_run', 'complete': False, 'reason': 'RUN_FULL_DATASETS is False'}
(RUN_DIR / 'full_status.json').write_text(json.dumps(FULL_REPORT, indent=2))
print(json.dumps(FULL_REPORT, indent=2))


In [ ]:
# Capacity and throughput only; repeated text does not prove retrieval quality.
RUN_CONTEXT_SCALING = True
if RUN_CONTEXT_SCALING:
    CONTEXT_REPORT = bench.run_full_length_1m_context_benchmark(
        model=model, tokenizer=tokenizer,
        horizons=(32768, 131072, 262144, 524288, 1048576),
        max_live_tokens=131072, gen_tokens=512, verbose=True,
    )
else:
    CONTEXT_REPORT = {'status': 'not_run', 'reason': 'RUN_CONTEXT_SCALING is False'}
(RUN_DIR / 'context_status.json').write_text(json.dumps(CONTEXT_REPORT, indent=2))
print(json.dumps(CONTEXT_REPORT, indent=2))


In [ ]:
# Generate a project example; preserve any existing personal models.yml.
from sparkx_omp_server import DEFAULT_MODEL_ID, generate_omp_models_yaml
OMP_CONFIG = generate_omp_models_yaml(
    output_path=str(RUN_DIR / 'omp_models.yml'), port=8000,
    model_id=DEFAULT_MODEL_ID, context_window=32768,
)
print(OMP_CONFIG)
RUN_SERVER = False
if RUN_SERVER:
    omp_server = bench.start_loop72_openai_compatible_server(model, tokenizer, port=8000, model_id=DEFAULT_MODEL_ID, background=True)
# In the same host/network namespace, run: python tools/omp_eval.py --help
# Use its actual OMP coding fixture; inspect raw events and the external test result.
# Colab localhost and laptop localhost are different; use a controlled tunnel if needed.


## Training and release sequence

Train mHC with native attention first and validate checkpoint persistence and train/eval parity (`SparkX_Loop72_MHC_Training_Colab.ipynb`). Fit per-visit D-LSA selectors against that frozen checkpoint (`Stage E`), using calibration data separate from evaluation. Quantized cache and extreme context extension follow the BF16 dense/sparse acceptance gates. The detailed recipe and upstream benchmark requirements are in `docs/roadmap/09_LOOP72_MHC_TRAINING_AND_VALIDATION.md`.
